In [14]:
from propp_fr import (
    load_spacy_model,
    generate_tokens_df,
    load_mentions_detection_model,
    load_tokenizer_and_embedding_model,
    get_embedding_tensor_from_tokens_df,
    generate_entities_df,
    add_features_to_entities,
    load_coreference_resolution_model,
    perform_coreference,
    extract_attributes,
    generate_characters_dict,
    load_text_file,
    clean_text,
    load_tokens_df,
    save_tokens_df,
    save_text_file,
    save_entities_df,
    load_entities_df,
)

import os
import torch
import joblib
import ast
from tqdm.auto import tqdm

import pandas as pd

In [15]:
detective_detector_model_path = "detective_detector_model.pkl"

detective_detector_dict = joblib.load(detective_detector_model_path)
detective_detector_model = detective_detector_dict["model"]
detective_detector_att_types = detective_detector_dict["att_types"]

print(detective_detector_att_types)
print(detective_detector_model)

['agent_embeddings', 'mod_embeddings', 'pos_embeddings']
SVC(C=1, class_weight='balanced', degree=2, gamma=1, probability=True,
    random_state=42)


In [16]:
annotated_detectives_dataset_path = os.path.join("data", "detectives_features_dataset.pt")
annotated_detectives_dataset = torch.load(annotated_detectives_dataset_path)

file_name_to_annotated_char_ids_map = {}
for char_dict in annotated_detectives_dataset:
    file_name = char_dict["file_name"]
    char_id = char_dict["char_id"]
    if file_name not in file_name_to_annotated_char_ids_map:
        file_name_to_annotated_char_ids_map[file_name] = []
    file_name_to_annotated_char_ids_map[file_name].append(char_id)

In [17]:
# How many characters per book should we keep ?

max_characters_to_keep = 10

In [18]:
all_tokens_entities_files_directory = "/home/antoine/Bureau/OUTPUT_BOOK_GEN_Z"
all_attributes_embeddings_directory = "/home/antoine/Bureau/detectives/attribute_embeddings"

extension = ".book"
book_files = sorted([f.replace(extension, "") for f in os.listdir(all_tokens_entities_files_directory) if f.endswith(extension)], reverse=False)


In [19]:
def load_book(file_path):
    characters_list = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()  # Remove any leading/trailing spaces or newline characters
            if line:  # Ensure the line is not empty
                characters_list.append(ast.literal_eval(line))  # Convert string to dict

    return characters_list

In [20]:
# overall_character_dataset = []
#
# for file_name in tqdm(book_files[:]):
#     attributes_embeddings_path = os.path.join(all_attributes_embeddings_directory, f"{file_name}.attribute_embeddings")
#     if not os.path.exists(attributes_embeddings_path):
#         print(f"No attributes embeddings found for {file_name}")
#         continue
#
#     attribute_embeddings = torch.load(attributes_embeddings_path)
#
#     tokens_df = load_tokens_df(file_name, all_tokens_entities_files_directory)
#     attributes_tokens_df = tokens_df[tokens_df["is_PER_attribute"] == 1].copy().reset_index(drop=True)
#
#     entities_df = load_entities_df(file_name, all_tokens_entities_files_directory)
#     entities_df = entities_df[entities_df["cat"]=="PER"]
#     characters_list = load_book(os.path.join(all_tokens_entities_files_directory, file_name + ".book"))
#
#     char_ids = sorted([character_list["id"] for character_list in characters_list])
#     char_ids = char_ids[:min(len(char_ids), max_characters_to_keep)]
#     if file_name in file_name_to_annotated_char_ids_map:
#         char_ids = char_ids + file_name_to_annotated_char_ids_map[file_name]
#     char_ids = list(set([int(char_id) for char_id in char_ids]))
#     if len(char_ids) < max_characters_to_keep:
#         print(char_ids)
#
#     for char_id in char_ids:
#         try: char_name = characters_list[char_id]["mentions"]["proper"][0]["n"]
#         except:
#             try: char_name = characters_list[char_id]["mentions"]["common"][0]["n"]
#             except: char_name = f"char_{char_id}"
#         plural_mention_ratio = characters_list[char_id]["number"]["inference"]["Plural"]
#         gender_ratio = characters_list[char_id]["gender"]["inference"]["Male"]
#
#         character_entities_df = entities_df[entities_df["COREF"] == char_id].copy()
#         character_head_ids = character_entities_df["head_id"].tolist()
#
#         attributes_ids = {
#             "agent":attributes_tokens_df[attributes_tokens_df["char_att_agent"].isin(character_head_ids)].index.tolist(),
#             "patient":attributes_tokens_df[attributes_tokens_df["char_att_patient"].isin(character_head_ids)].index.tolist(),
#             "mod":attributes_tokens_df[attributes_tokens_df["char_att_mod"].isin(character_head_ids)].index.tolist(),
#             "pos":attributes_tokens_df[attributes_tokens_df["char_att_poss"].isin(character_head_ids)].index.tolist(),
#         }
#
#         all_attributes_embeddings = []
#         for att_type in detective_detector_att_types:
#             att_type = att_type.split("_")[0]
#             all_attributes_embeddings.extend(attribute_embeddings[attributes_ids[att_type]])
#
#         if len(all_attributes_embeddings) == 0:
#             continue
#
#         all_attributes_embeddings = torch.stack(all_attributes_embeddings)
#         mean_embedding = torch.mean(all_attributes_embeddings, dim=0)
#         mean_embedding = torch.nan_to_num(mean_embedding, nan=0.0)
#
#         overall_character_dataset.append(
#             {
#                 "file_name": file_name,
#                 "char_name": char_name,
#                 "char_id": char_id,
#                 "mention_count":len(character_entities_df),
#                 "mention_ratio":len(character_entities_df)/len(entities_df[entities_df["cat"]=="PER"]),
#                 "attributes_count":len(all_attributes_embeddings),
#                 "plural_ratio":plural_mention_ratio,
#                 "gender_ratio":gender_ratio,
#                 "agent_lemmas": attributes_tokens_df.loc[attributes_ids["agent"], "lemma"].tolist(),
#                 "patient_lemmas": attributes_tokens_df.loc[attributes_ids["patient"], "lemma"].tolist(),
#                 "mod_lemmas": attributes_tokens_df.loc[attributes_ids["mod"], "lemma"].tolist(),
#                 "pos_lemmas": attributes_tokens_df.loc[attributes_ids["pos"], "lemma"].tolist(),
#                 "mean_embedding": mean_embedding,
#                 # "patient_embeddings": attribute_embeddings[patient_attributes_ids],
#                 # "mod_embeddings": attribute_embeddings[mod_attributes_ids],
#                 # "pos_embeddings": attribute_embeddings[pos_attributes_ids],
#              })
#
# # ✅ Save it
# overall_character_dataset_path = os.path.join("data", "overall_character_dataset.pt")
# torch.save(overall_character_dataset, overall_character_dataset_path)

In [21]:
overall_character_dataset_path = os.path.join("data", "overall_character_dataset.pt")

# ✅ Load it back
overall_character_dataset = torch.load(overall_character_dataset_path)
print(overall_character_dataset[0])

{'file_name': '1811_Chateaubriand-François-Rene-de_Oeuvres-completes', 'char_name': 'm. fauvel', 'char_id': 0, 'mention_count': 710, 'mention_ratio': 0.06448683015440508, 'attributes_count': 497, 'plural_ratio': 0.0, 'gender_ratio': 0.96, 'agent_lemmas': ['retirer', 'pouvoir', 'entendre', 'pouvoir', 'aller', 'suivre', 'obstiner', 'croire', 'voir', 'citer', 'devoir', 'voir', 'écrire', 'venir', 'saluer', 'savoir', 'avoir', 'recevoir', 'remettre', 'vouloir', 'renvoyer', 'donner', 'pouvoir', 'faire', 'aller', 'rendre', 'revenir', 'prendre', 'avoir', 'parler', 'aimer', 'parcourir', 'console', 'donner', 'savoir', 'voir', 'découvrir', 'pouvoir', 'croire', 'quitter', 'donner', 'embarquer', 'amuser', 'voir', 'accepter', 'continuer', 'savoir', 'assurer', 'traverser', 'passer', 'convaincre', 'voir', 'passer', 'défier', 'avoir', 'jeter', 'vouloir', 'abandonner', 'aimer', 'entendre', 'faire', 'aller', 'répondre', 'aller', 'rencontrer', 'savoir', 'passer', 'croire', 'trouver', 'chercher', 'apporter'

In [22]:
print(len(overall_character_dataset))
unique = []
seen = set()

for d in overall_character_dataset:
    key = (d['file_name'], d['char_id'])
    if key not in seen:
        seen.add(key)
        unique.append(d)
print(len(unique))

29849
29849


In [23]:
annotated_char_gold_label_map = {}
for char_dict in annotated_detectives_dataset:
    file_name = char_dict["file_name"]
    char_id = char_dict["char_id"]
    gold_label = char_dict["label"]
    if file_name not in annotated_char_gold_label_map:
        annotated_char_gold_label_map[file_name] = {}
    annotated_char_gold_label_map[file_name][char_id] = gold_label

In [24]:
for dict_index, char_dict in enumerate(overall_character_dataset):
    file_name = char_dict["file_name"]
    if file_name in file_name_to_annotated_char_ids_map:
        char_id = char_dict["char_id"]
        if char_id in file_name_to_annotated_char_ids_map[file_name]:
            char_dict["gold_label"] = annotated_char_gold_label_map[file_name][char_id]

In [25]:
X = [char_dict["mean_embedding"] for char_dict in overall_character_dataset]
predicted_label = detective_detector_model.predict(X)

In [26]:
from collections import Counter

Counter(predicted_label)

Counter({0: 28814, 1: 1035})

In [32]:
import pandas as pd

article_dataset_label_mapping = {}
chapitre_characters_df = pd.read_csv('data/all_characters_SVC_predictions_all_attributes.csv')
for file_name, char_id, label in chapitre_characters_df[["file_name", "char_id", "label"]].values:
    if file_name not in article_dataset_label_mapping:
        article_dataset_label_mapping[file_name] = {}
    article_dataset_label_mapping[file_name][char_id] = label

In [33]:
chapitre_characters_df

,id,file_name,char_id,char_name,mention_count,mention_ratio,attributes_count,plural_ratio,gender_ratio,gold_label,...,author,author_character,multi_doc_character,char_att_agent,char_att_patient,char_att_mod,char_att_poss,all_attributes,cluster,label
0,0,1811_Chateaubriand-François-Rene-de_Oeuvres-co...,0,m. fauvel,710,0.064487,497,0.00,0.96,NaN,...,Chateaubriand-François-Rene-de,Chateaubriand-François-Rene-de_m. fauvel,other,retirer pouvoir entendre pouvoir aller suivre ...,recevoir conduire rendre reconnaître crier voi...,cher cher jeune français premier français bien...,équipage janissaire guide guide consolation ja...,retirer pouvoir entendre pouvoir aller suivre ...,NaN,0
1,1,1811_Chateaubriand-François-Rene-de_Oeuvres-co...,1,djezzar,483,0.043869,369,0.00,1.00,NaN,...,Chateaubriand-François-Rene-de,Chateaubriand-François-Rene-de_djezzar,other,vouloir venir mourir céder commettre finir com...,voir abandonner accompagner accabler mettre at...,confus aise vaste voyageur debout ému espagnol...,ordre merveille poëte philosophe natte place t...,vouloir venir mourir céder commettre finir com...,NaN,0
2,2,1811_Chateaubriand-François-Rene-de_Oeuvres-co...,2,chandler,426,0.038692,315,0.00,1.00,NaN,...,Chateaubriand-François-Rene-de,Chateaubriand-François-Rene-de_chandler,other,monter décrire raconter arriver rendre trouver...,aimer avoir gravir rejoindre attendre chercher...,espagnol lâche obscur étranger fatigué grand a...,destinée nom sort jour atticus nation enfant c...,monter décrire raconter arriver rendre trouver...,NaN,0
3,3,1811_Chateaubriand-François-Rene-de_Oeuvres-co...,3,le père babin,302,0.027430,231,0.00,1.00,0.0,...,Chateaubriand-François-Rene-de,Chateaubriand-François-Rene-de_le père babin,other,dire savoir voir pouvoir avouer découvrir voir...,écouter écouter obliger voir vouloir parler vo...,autrichien aise brutal,temps ami revue tableau peinture route but rai...,dire savoir voir pouvoir avouer découvrir voir...,NaN,0
4,4,1811_Chateaubriand-François-Rene-de_Oeuvres-co...,4,le fils d' ulysse,277,0.025159,215,0.00,1.00,NaN,...,Chateaubriand-François-Rene-de,Chateaubriand-François-Rene-de_le fils d' ulysse,other,faire devoir commencer obstiner croire essayer...,entendre attendre soutenir assiéger accompagne...,obscur fils xiii simple,étude projet attention embarras cicérone grec ...,faire devoir commencer obstiner croire essayer...,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29860,29860,2020_Springora-Vanessa_Le-consentement,5,julien et moi,47,0.010410,29,0.89,1.00,NaN,...,Springora-Vanessa,Springora-Vanessa_julien et moi,other,dormir amasser désigner allier ressentir chama...,rejoindre ravir tendre,fiancé tout,tente matelas lèvre matelas corps imagination ...,dormir amasser désigner allier ressentir chama...,NaN,0
29861,29861,2020_Springora-Vanessa_Le-consentement,6,deux personnes,27,0.005980,24,0.39,0.90,NaN,...,Springora-Vanessa,Springora-Vanessa_deux personnes,other,faire porter avoir répondre dire sortir approc...,écouter,barbu,lèvre voix parole heure bras collègue ton entr...,faire porter avoir répondre dire sortir approc...,NaN,0
29862,29862,2020_Springora-Vanessa_Le-consentement,7,ma mère,24,0.005316,20,0.00,0.00,NaN,...,Springora-Vanessa,Springora-Vanessa_ma mère,other,attendre avoir risque pouvoir jeter croire dir...,embobiner fixer,NaN,conscience responsabilité fille visage collègu...,attendre avoir risque pouvoir jeter croire dir...,NaN,0
29863,29863,2020_Springora-Vanessa_Le-consentement,8,char_8,21,0.004651,16,1.00,0.00,NaN,...,Springora-Vanessa,Springora-Vanessa_char_8,other,correspondre voir déposséder retrouver avoir,rejoindre,seul sensible,relation amour âge langue mot rencontre histoi...,correspondre voir déposséder retrouver avoir r...,NaN,0


In [34]:
annotated_char_gold_label_map = {}
for char_dict in overall_character_dataset:
    file_name = char_dict["file_name"]
    if file_name in article_dataset_label_mapping:
        char_id = char_dict["char_id"]
        if char_id in article_dataset_label_mapping[file_name]:
            char_dict["label"] = article_dataset_label_mapping[file_name][char_id]

In [37]:
labels = [char_dict["label"] for char_dict in overall_character_dataset if 'label' in char_dict]
print(Counter(labels), Counter(predicted_label))

Counter({0: 29136, 1: 713}) Counter({0: 28814, 1: 1035})


In [38]:
print(len(predicted_label), len(labels))

29849 29849


In [42]:
multi_doc_characters = [
    {"character_name": "le_comissaire_maigret", "authors": ["Simenon-Georges"], "aliases": ["maigret", "le commissaire maigret", "monsieur maigret"]},
    {"character_name": "juve", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["juve"]},
    {"character_name": "rouletabille", "authors": ["Leroux-Gaston"], "aliases": ["rouletabille"]},
    {"character_name": "fantômas", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["fantômas"]},
    {"character_name": "jérôme_fandor", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["fandor", "jérôme fandor"]},
    {"character_name": "arsène_lupin", "authors": ["Leblanc-Maurice"], "aliases": ["arsène lupin", "lupin", "raoul", "jim barnett", "barnett", "don luis", "victor"]},
    {"character_name": "adamsberg", "authors": ["Vargas-Fred"], "aliases": ["adamsberg"]},
    {"character_name": "monsieur_lecoq", "authors": ["Gaboriau-Emile"], "aliases": ["lecoq", "m. lecoq", "m. verduret"]},
    {"character_name": "inspecteur_ganimard", "authors": ["Leblanc-Maurice"], "aliases": ["ganimard"]},
    {"character_name": "inspecteur_cadin", "authors": ["Daeninckx-Didier"], "aliases": ["cadin"]},
    {"character_name": "gabriel_lecouvreur", "authors": ["Daeninckx-Didier"], "aliases": ["gabriel"]},
    {"character_name": "camille", "authors": ["Lemaitre-Pierre"], "aliases": ["camille"]},
    {"character_name": "inspecteur_ménardier", "authors": ["Bernede-Arthur"], "aliases": ["ménardier"]},
    {"character_name": "inspecteur_huret", "authors": ["Bouvier-Alexis"], "aliases": ["l' agent huret"]},
    {"character_name": "chéri_bibi", "authors": ["Leroux-Gaston"], "aliases": ["chéri - bibi"]},
    {"character_name": "juge_denizet", "authors": ["Zola-Emile"], "aliases": ["m. denizet"]},
    {"character_name": "herlock_sholmes", "authors": ["Leblanc-Maurice"], "aliases": ["sholmès"]},
    {"character_name": "isidore_beautrelet", "authors": ["Leblanc-Maurice"], "aliases": ["isidore"]},
    {"character_name": "prasville_secrétaire_général_préfecture", "authors": ["Leblanc-Maurice"], "aliases": ["prasville"]},
    {"character_name": "père_tabaret", "authors": ["Gaboriau-Emile"], "aliases": ["le père tabaret"]},
    {"character_name": "père_plantat", "authors": ["Gaboriau-Emile"], "aliases": ["le père plantat"]},
    {"character_name": "nestor_burma", "authors": ["Malet-Leo"], "aliases": ["monsieur burma", "nestor burma"]},
    {"character_name": "john_jarvis", "authors": ["Le-Rouge-Gustave"], "aliases": ["john_jarvis"]},
    {"character_name": "wallas", "authors": ["Robbe-Grillet-Alain"], "aliases": ["wallas"]},
    {"character_name": "san_antonio", "authors": ["San-Antonio"], "aliases": ["san-antonio"]},
    {"character_name": "jobin", "authors": ["Montepin-Xavier-de"], "aliases": ["jobin"]},
    {"character_name": "commissaire_niémans", "authors": ["Grange-Jean-Christophe"], "aliases": ["niémans"]},
    {"character_name": "john_jarvis", "authors": ["Le-Rouge-Gustave"], "aliases": ["john jarvis"]},
    {"character_name": "théodore_béchoux", "authors": ["Leblanc-Maurice"], "aliases": ["béchoux"]},
    {"character_name": "monsieur_desmalions", "authors": ["Leblanc-Maurice"], "aliases": ["m. desmalions"]},
    {"character_name": "monsieur_hilaire", "authors": ["Leroux-Gaston"], "aliases": ["m. hilaire"]},
    {"character_name": "serge_rénine", "authors": ["Leblanc-Maurice"], "aliases": ["rénine"]},
    {"character_name": "hercule_petitgris", "authors": ["Leblanc-Maurice"], "aliases": ["petitgris"]},
    {"character_name": "patricia_johnston", "authors": ["Leblanc-Maurice"], "aliases": ["patricia"]},
    {"character_name": "monsieur_bessières", "authors": ["Leroux-Gaston"], "aliases": ["m. bessières"]},
    {"character_name": "monsieur_lebouc", "authors": ["Leroux-Gaston"], "aliases": ["lebouc"]},
    {"character_name": "cravely", "authors": ["Leroux-Gaston"], "aliases": ["cravely"]},
    {"character_name": "commissaire_delvigne", "authors": ["Simenon-Georges"], "aliases": ["le commissaire delvigne"]},
    {"character_name": "inspecteur_boutigues", "authors": ["Simenon-Georges"], "aliases": ["boutigues"]},
    {"character_name": "torrence", "authors": ["Simenon-Georges"], "aliases": ["torrence"]},
    {"character_name": "richard_lafargue", "authors": ["Jonquet-Thierry"], "aliases": ["richard"]},
    {"character_name": "rené_griffon", "authors": ["Daeninckx-Didier"], "aliases": ["griffon"]},
    {"character_name": "alain_deligny", "authors": ["Daeninckx-Didier"], "aliases": ["alain deligny"]},
    {"character_name": "michèle_fogel", "authors": ["Daeninckx-Didier"], "aliases": ["michèle fogel"]},
    {"character_name": "yves guyot", "authors": ["Daeninckx-Didier"], "aliases": ["guyot", "yves guyot"]},
    {"character_name": "commissaire_divisionnaire_darqué", "authors": ["Daeninckx-Didier"], "aliases": ["darqué"]},
    {"character_name": "inspecteur_divisionnaire_darqué", "authors": ["Jonquet-Thierry"], "aliases": ["rovère"]},
    {"character_name": "colonel_starr", "authors": ["Gary-Romain"], "aliases": ["starr"]},
    {"character_name": "monsignor_domani", "authors": ["Gary-Romain"], "aliases": ["monsignor domani"]},
    {"character_name": "adrien_danglard", "authors": ["Vargas-Fred"], "aliases": ["danglard"]},
    {"character_name": "bouzille", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["bouzille"]},
]

char_name_to_multidoc_mapping = {}

for multidoc_character in multi_doc_characters:
    multidoc_alias = multidoc_character["character_name"]
    for author in multidoc_character["authors"]:
        for alias in multidoc_character["aliases"]:
            char_name_to_multidoc_mapping[f"{author} {alias}"] = multidoc_alias

for char_dict in overall_character_dataset:
    file_name = char_dict["file_name"]
    year = int(file_name.split("_")[0])
    author = file_name.split("_")[1]
    char_name = char_dict["char_name"]
    author_alias = f"{author} {char_name}"
    if author_alias in char_name_to_multidoc_mapping:
        multidoc_alias = char_name_to_multidoc_mapping[author_alias]
    else:
        multidoc_alias = author_alias
    char_dict["year"] = year
    char_dict["author"] = author
    char_dict["multidoc_alias"] = multidoc_alias

In [43]:
Counter([char_dict["multidoc_alias"] for char_dict in overall_character_dataset])

Counter({'le_comissaire_maigret': 90,
         'San-Antonio béru': 59,
         'jérôme_fandor': 32,
         'juve': 31,
         'San-Antonio pinaud': 30,
         'San-Antonio le gros': 30,
         'Simenon-Georges lucas': 25,
         'arsène_lupin': 24,
         'San-Antonio félicie': 24,
         'San-Antonio bérurier': 24,
         'fantômas': 22,
         'Simenon-Georges mme maigret': 19,
         'San-Antonio pinuche': 18,
         'Simenon-Georges janvier': 17,
         'Allain-Marcel-&-Souvestre-Pierre hélène': 14,
         'san_antonio': 14,
         'Ponson-du-Terrail-Pierre rocambole': 13,
         'San-Antonio le vieux': 12,
         'Balzac-Honore-de char_9': 11,
         'Simenon-Georges lapointe': 11,
         'San-Antonio berthe': 11,
         'Rolland-Romain christophe': 10,
         'Zevaco-Michel pardaillan': 10,
         'Dumas-Alexandre henri': 9,
         'San-Antonio pépère': 9,
         'Loti-Pierre char_8': 8,
         'Allain-Marcel-&-Souvestre-Pierre lad

In [44]:
print(f"Character count: {len(overall_character_dataset)}")
print(f"Character count: {overall_character_dataset[0].keys()}")

Character count: 29849
Character count: dict_keys(['file_name', 'char_name', 'char_id', 'mention_count', 'mention_ratio', 'attributes_count', 'plural_ratio', 'gender_ratio', 'agent_lemmas', 'patient_lemmas', 'mod_lemmas', 'pos_lemmas', 'mean_embedding', 'label', 'year', 'author', 'multidoc_alias'])


In [45]:
torch.save(overall_character_dataset, overall_character_dataset_path)